In [ ]:
!pip install pyarrow


In [ ]:
import pandas as pd

In [ ]:
import pickle

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [ ]:
mean_squared_error

In [27]:
import os, mlflow, pathlib

mlflow.set_experiment("nyc-taxi-experiment")
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_registry_uri("sqlite:///mlflow.db")


In [ ]:
pd.__version__

In [ ]:
def read_dataframe(filename):

    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [ ]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')


In [ ]:
len(df_train), len(df_val)

In [ ]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [ ]:
df_val

In [33]:
categorical = ['PU_DO']   #  ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)


In [ ]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

#mean_squared_error(y_val, y_pred, squared=False)
root_mean_squared_error(y_val, y_pred)

In [ ]:
mlflow.sklearn.autolog(log_datasets=False)

In [ ]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [ ]:
with mlflow.start_run():
    
    mlflow.set_tag("developer","eriton")
    
    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    
    mlflow.log_metric("rmse", rmse)
    
    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [ ]:
import xgboost as xgb

In [ ]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [ ]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [ ]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

In [28]:
params = { 
    'max_depth': 41,
    'learning_rate': 0.08048724945407973,
    'reg_alpha': 0.011688402031886633,
    'reg_lambda': 0.004836768546153373,
    'min_child_weight': 1.2306633622264325,
    'objective': 'reg:linear',
    'seed': 42
}

In [29]:
mlflow.xgboost.autolog()

booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )

2026/04/07 22:54:31 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '782b73d705ad49448af70a542692b4f5', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow


/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [22:54:33] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.56274
[1]	validation-rmse:10.97980
[2]	validation-rmse:10.45918
[3]	validation-rmse:9.99309
[4]	validation-rmse:9.58068
[5]	validation-rmse:9.21196
[6]	validation-rmse:8.88714
[7]	validation-rmse:8.60014
[8]	validation-rmse:8.34798
[9]	validation-rmse:8.12540
[10]	validation-rmse:7.92995
[11]	validation-rmse:7.75993
[12]	validation-rmse:7.60817
[13]	validation-rmse:7.47737
[14]	validation-rmse:7.36125
[15]	validation-rmse:7.26119
[16]	validation-rmse:7.17201
[17]	validation-rmse:7.09353
[18]	validation-rmse:7.02673
[19]	validation-rmse:6.96606
[20]	validation-rmse:6.91461
[21]	validation-rmse:6.86840
[22]	validation-rmse:6.82805
[23]	validation-rmse:6.79238
[24]	validation-rmse:6.76135
[25]	validation-rmse:6.73299
[26]	validation-rmse:6.70765
[27]	validation-rmse:6.68506
[28]	validation-rmse:6.66566
[29]	validation-rmse:6.64737
[30]	validation-rmse:6.63176
[31]	validation-rmse:6.61735
[32]	validation-rmse:6.60405
[33]	validation-rmse:6.59233
[34]	validation-rmse:

2026/04/07 22:57:20 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/04/07 22:57:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [30]:
logged_model = 'runs:/782b73d705ad49448af70a542692b4f5/model'

In [31]:
# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)


   

In [32]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: /workspaces/mlops-zoomcamp/03 - training/experiment_tracking/mlruns/1/models/m-4209f05d9cd041a5b558982c2e976547/artifacts
  flavor: mlflow.xgboost
  run_id: 782b73d705ad49448af70a542692b4f5

In [34]:
valid = xgb.DMatrix(X_val, label=y_val)

In [36]:
loaded_model.predict(X_val)

array([14.527894 ,  7.1385155, 15.74355  , ..., 13.456723 ,  6.3960595,
        8.215588 ], shape=(61921,), dtype=float32)

In [37]:
xgb_model = mlflow.xgboost.load_model(logged_model)
xgb_model

In [38]:
xgb_model.predict(valid)

array([14.527894 ,  7.1385155, 15.74355  , ..., 13.456723 ,  6.3960595,
        8.215588 ], shape=(61921,), dtype=float32)

In [44]:
mlflow.xgboost.autolog(disable=True)
mlflow.set_experiment("my-cool-experiment")
with mlflow.start_run():
# Hyperparameter for run 09923bbad64045ca837a1656254ce756
    params = { 
        'max_depth': 41,
        'learning_rate': 0.08048724945407973,
        'reg_alpha': 0.011688402031886633,
        'reg_lambda': 0.004836768546153373,
        'min_child_weight': 1.2306633622264325,
        'objective': 'reg:linear',
        'seed': 42
    }
    mlflow.log_params(params)
    
    booster = xgb.train(
                params=params,
                dtrain=train,
                num_boost_round=1000,
                evals=[(valid, 'validation')],
                early_stopping_rounds=50
            )
    # get the y_pred from X_train
    y_pred = booster.predict(valid)
    # get RMSE and record on mlflow
    rmse = round(root_mean_squared_error(y_val, y_pred),2)
    print("RMSE for training data:", rmse)
    mlflow.log_metric("rmse", rmse)
    # log xgboost model to mlflow
    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")
    # log the preprocessor DictVectorizer
    with open("models/preprocessor.bin", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.bin", artifact_path="preprocessor")

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [23:53:52] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.56274
[1]	validation-rmse:10.97980
[2]	validation-rmse:10.45918
[3]	validation-rmse:9.99309
[4]	validation-rmse:9.58068
[5]	validation-rmse:9.21196
[6]	validation-rmse:8.88714
[7]	validation-rmse:8.60014
[8]	validation-rmse:8.34798
[9]	validation-rmse:8.12540
[10]	validation-rmse:7.92995
[11]	validation-rmse:7.75993
[12]	validation-rmse:7.60817
[13]	validation-rmse:7.47737
[14]	validation-rmse:7.36125
[15]	validation-rmse:7.26119
[16]	validation-rmse:7.17201
[17]	validation-rmse:7.09353
[18]	validation-rmse:7.02673
[19]	validation-rmse:6.96606
[20]	validation-rmse:6.91461
[21]	validation-rmse:6.86840
[22]	validation-rmse:6.82805
[23]	validation-rmse:6.79238
[24]	validation-rmse:6.76135
[25]	validation-rmse:6.73299
[26]	validation-rmse:6.70765
[27]	validation-rmse:6.68506
[28]	validation-rmse:6.66566
[29]	validation-rmse:6.64737
[30]	validation-rmse:6.63176
[31]	validation-rmse:6.61735
[32]	validation-rmse:6.60405
[33]	validation-rmse:6.59233
[34]	validation-rmse:

2026/04/07 23:56:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE for training data: 6.31
